# MSR-Action3D: Feature Extraction (Depth + Skeleton)

Extracts features from MSR-Action3D dataset using the **identical** feature extraction
pipeline as UTD-MHAD, but only for depth and skeleton modalities (no RGB video, no inertial).

**MSR-Action3D dataset:**
- 20 action types, 10 subjects, 2-3 trials each
- 567 depth map sequences (640×240 resolution)
- Skeleton: 20 joints (same Kinect v1 layout as UTD-MHAD)

**File formats:**
- Depth: `a{action}_s{subject}_e{trial}_sdepth.bin` (binary: header + depth frames)
- Skeleton: `a{action}_s{subject}_e{trial}_skeleton.txt` (flat text: n_frames × 20 joints × 4 values)

**Output:** Saves features in `features/` folder in the same format as UTD-MHAD.

In [1]:
import os
import sys
import glob
import struct
import warnings
import time
import numpy as np
from scipy import stats, signal
from scipy.ndimage import convolve
from sklearn.preprocessing import LabelEncoder
import logging
import joblib

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
class Config:
    """Central configuration for MSR-Action3D."""

    # Dataset paths (update these to your MSR-Action3D location)
    DEPTH_DIR = "Depth"
    SKELETON_DIR = "MSRAction3DSkeleton(20joints)"

    # Output directory for saved features
    FEATURES_DIR = "features_new"

    # Dataset properties
    NUM_ACTIONS = 20
    NUM_SUBJECTS = 10
    NUM_JOINTS = 20  # Same Kinect v1 skeleton as UTD-MHAD

    # MSR-Action3D action names (1-indexed)
    ACTION_NAMES = [
        'high_arm_wave', 'horizontal_arm_wave', 'hammer', 'hand_catch',
        'forward_punch', 'high_throw', 'draw_x', 'draw_tick',
        'draw_circle', 'hand_clap', 'two_hand_wave', 'side_boxing',
        'bend', 'forward_kick', 'side_kick', 'jogging',
        'tennis_swing', 'tennis_serve', 'golf_swing', 'pickup_throw'
    ]

    # Depth feature extraction parameters
    DEPTH_SAMPLE_FRAMES = 10

    # Which modalities to use
    USE_SKELETON = True
    USE_DEPTH = True

config = Config()
print(f"Depth dir: {config.DEPTH_DIR}")
print(f"Skeleton dir: {config.SKELETON_DIR}")
print(f"Output: {config.FEATURES_DIR}")
print(f"Actions: {config.NUM_ACTIONS}, Subjects: {config.NUM_SUBJECTS}")

Depth dir: Depth
Skeleton dir: MSRAction3DSkeleton(20joints)
Output: features_new
Actions: 20, Subjects: 10


## Data Loading

In [3]:
class MSRAction3DLoader:
    """
    Loads MSR-Action3D dataset from .bin depth files and .txt skeleton files.

    File naming: a{action:02d}_s{subject:02d}_e{episode:02d}_sdepth.bin
                 a{action:02d}_s{subject:02d}_e{episode:02d}_skeleton.txt
    """

    def __init__(self, config):
        self.config = config

    def _parse_filename(self, filename):
        """Extract action, subject, episode from filename."""
        base = os.path.basename(filename)
        parts = base.split('_')
        action = int(parts[0][1:])
        subject = int(parts[1][1:])
        episode = int(parts[2][1:])
        return action, subject, episode

    def load_depth_data(self):
        """
        Load depth .bin files.
        MSR-Action3D depth format:
          - Header: nframes(4B int32), ncols(4B int32), nrows(4B int32)
          - Per frame: nrows × ncols depth values (4B int32 each, but often stored
            with the top 3 bits as player index and lower 13 bits as depth)
        Returns: list of (action, subject, episode, depth_array)
        """
        files = sorted(glob.glob(os.path.join(self.config.DEPTH_DIR, "a*_s*_e*_sdepth.bin")))

        if not files:
            logger.warning(f"No depth files found in {self.config.DEPTH_DIR}")
            return []

        data = []
        for f in files:
            action, subject, episode = self._parse_filename(f)
            try:
                depth_seq = self._read_depth_bin(f)
                if depth_seq is not None and depth_seq.shape[0] > 0:
                    data.append((action, subject, episode, depth_seq))
            except Exception as e:
                logger.debug(f"Error loading {f}: {e}")

        logger.info(f"Loaded {len(data)} depth sequences")
        return data

    def _read_depth_bin(self, filepath):
        """
        Read MSR-Action3D binary depth file.
        Format: [nframes(i4), ncols(i4), nrows(i4), then nframes * nrows * ncols int32 values]
        The depth values encode player index in top bits; we extract just the depth.
        Returns: (nframes, nrows, ncols) numpy array
        """
        with open(filepath, 'rb') as f:
            # Read header
            header = f.read(12)
            if len(header) < 12:
                return None
            nframes, ncols, nrows = struct.unpack('iii', header)

            if nframes <= 0 or ncols <= 0 or nrows <= 0:
                return None
            if nframes > 1000 or ncols > 1000 or nrows > 1000:
                return None

            # Read all depth data
            total_pixels = nframes * nrows * ncols
            raw = np.fromfile(f, dtype=np.int32, count=total_pixels)

            if len(raw) < total_pixels:
                # File might be truncated; use what we have
                usable_frames = len(raw) // (nrows * ncols)
                if usable_frames == 0:
                    return None
                raw = raw[:usable_frames * nrows * ncols]
                nframes = usable_frames

            depth = raw.reshape(nframes, nrows, ncols)

            # Extract depth from lower 13 bits (MSR format packs player index in upper bits)
            depth = depth & 0x1FFF
            depth = depth.astype(np.float64)

            return depth  # (nframes, nrows, ncols)

    def load_skeleton_data(self):
        """
        Load skeleton .txt files.
        MSR-Action3D skeleton (20 joints) format:
          Each file has n_frames * 20 * 4 float values in a flat text file.
          Per joint: u, v, d, c (screen coords u,v; depth d; confidence c)
          We use (u, v, d) as pseudo-3D coordinates for feature extraction.
        Returns: list of (action, subject, episode, skeleton_array)
        """
        files = sorted(glob.glob(os.path.join(self.config.SKELETON_DIR, "a*_s*_e*_skeleton.txt")))

        if not files:
            logger.warning(f"No skeleton files found in {self.config.SKELETON_DIR}")
            return []

        data = []
        for f in files:
            action, subject, episode = self._parse_filename(f)
            try:
                skel = self._read_skeleton_txt(f)
                if skel is not None and skel.shape[0] > 0:
                    data.append((action, subject, episode, skel))
            except Exception as e:
                logger.debug(f"Error loading {f}: {e}")

        logger.info(f"Loaded {len(data)} skeleton sequences")
        return data

    def _read_skeleton_txt(self, filepath):
        """
        Read MSR-Action3D skeleton text file.
        Format: flat list of floats, n_frames * 20 joints * 4 values (u, v, d, c).
        Returns: (20, 3, n_frames) array — same layout expected by SkeletonFeatureExtractor.
                 We use (u, v, d) as the 3 coordinates (dropping confidence c).
        """
        try:
            values = np.loadtxt(filepath)
        except Exception:
            return None

        if values.size == 0:
            return None

        values = values.flatten()
        n_values = len(values)
        n_per_frame = 20 * 4  # 20 joints × 4 values each
        n_frames = n_values // n_per_frame

        if n_frames == 0:
            return None

        # Trim to exact multiple
        values = values[:n_frames * n_per_frame]
        values = values.reshape(n_frames, 20, 4)

        # Extract (u, v, d) — drop confidence column
        skel = values[:, :, :3]  # (n_frames, 20, 3)

        u = skel[:, :, 0]  # screen x (pixels, ~0-640)
        v = skel[:, :, 1]  # screen y (pixels, ~0-240)  
        d = skel[:, :, 2]  # depth (mm)

        # Convert screen coords to pseudo-world coords (pinhole camera model)
        # Using approximate Kinect v1 intrinsics
        fx, fy = 525.0, 525.0  # focal length in pixels
        cx, cy = 320.0, 240.0  # principal point

        x_world = (u - cx) * d / fx
        y_world = (v - cy) * d / fy
        z_world = d

        skel = np.stack([x_world, y_world, z_world], axis=2)  # (n_frames, 20, 3)

        # Transpose to (20, 3, n_frames) to match UTD-MHAD format
        # (SkeletonFeatureExtractor.extract() does: skel = np.transpose(skel, (2, 0, 1)))
        skel = skel.transpose(1, 2, 0)  # (20, 3, n_frames)

        return skel

loader = MSRAction3DLoader(config)
print("Data loader defined")

Data loader defined


## Skeleton Feature Extraction (identical to UTD-MHAD)

In [4]:
class SkeletonFeatureExtractor:
    """
    Extracts features from skeleton joint sequences.
    IDENTICAL to UTD-MHAD pipeline — same joint indices, same features.
    """

    HIP_CENTER = 0
    SPINE = 1
    SHOULDER_CENTER = 2
    HEAD = 3

    UPPER_BODY = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
    LOWER_BODY = [0, 12, 13, 14, 15, 16, 17, 18, 19]
    HANDS = [7, 11]
    FEET = [15, 19]
    ENDPOINTS = [3, 7, 11, 15, 19]

    JOINT_PAIRS = [
        (7, 11), (15, 19), (7, 0), (11, 0), (7, 3), (11, 3),
        (15, 0), (19, 0), (6, 10), (5, 9), (4, 8), (13, 17),
    ]

    ANGLE_TRIPLETS = [
        (4, 5, 6), (8, 9, 10), (12, 13, 14), (16, 17, 18),
        (4, 2, 8), (5, 4, 2), (9, 8, 2), (0, 1, 2),
    ]

    def _normalize_skeleton(self, skel):
        hip = skel[:, self.HIP_CENTER:self.HIP_CENTER+1, :]
        skel_norm = skel - hip
        torso_vec = skel_norm[:, self.SHOULDER_CENTER, :] - skel_norm[:, self.HIP_CENTER, :]
        torso_len = np.linalg.norm(torso_vec, axis=1, keepdims=True)
        torso_len = np.clip(torso_len, 1e-6, None)
        skel_norm = skel_norm / torso_len[:, np.newaxis, :]
        return skel_norm

    def _compute_joint_distances(self, skel):
        distances = []
        for j1, j2 in self.JOINT_PAIRS:
            dist = np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1)
            distances.append(dist)
        return np.array(distances).T

    def _compute_joint_angles(self, skel):
        angles = []
        for j1, j2, j3 in self.ANGLE_TRIPLETS:
            v1 = skel[:, j1, :] - skel[:, j2, :]
            v2 = skel[:, j3, :] - skel[:, j2, :]
            cos_angle = np.sum(v1 * v2, axis=1) / (
                np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8)
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angles.append(np.arccos(cos_angle))
        return np.array(angles).T

    def _compute_velocity(self, skel):
        if skel.shape[0] < 2:
            return np.zeros_like(skel)
        vel = np.diff(skel, axis=0)
        vel = np.vstack([vel, vel[-1:]])
        return vel

    def _compute_acceleration(self, skel):
        vel = self._compute_velocity(skel)
        if vel.shape[0] < 2:
            return np.zeros_like(vel)
        acc = np.diff(vel, axis=0)
        acc = np.vstack([acc, acc[-1:]])
        return acc

    def _temporal_statistics(self, signal_2d):
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s), np.std(s), np.sqrt(np.mean(s**2)),
                stats.skew(s) if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s), np.median(s),
                np.percentile(s, 25), np.percentile(s, 75),
            ])
        return np.array(features)

    def _covariance_features(self, skel):
        flat = skel.reshape(skel.shape[0], -1)
        key_joints = [0, 1, 2, 3, 7, 11, 15, 19]
        key_indices = []
        for j in key_joints:
            key_indices.extend([j*3, j*3+1, j*3+2])
        flat_key = flat[:, key_indices]
        if flat_key.shape[0] < 2:
            cov_mat = np.zeros((flat_key.shape[1], flat_key.shape[1]))
        else:
            cov_mat = np.cov(flat_key.T)
        return cov_mat[np.triu_indices(cov_mat.shape[0])]

    def extract(self, skel):
        if skel is None or skel.shape[0] == 0:
            return None
        if skel.ndim != 3:
            return None

        # Convert from (20, 3, frames) to (frames, 20, 3)
        skel = np.transpose(skel, (2, 0, 1))
        skel = np.nan_to_num(skel, nan=0.0, posinf=0.0, neginf=0.0)

        skel_norm = self._normalize_skeleton(skel)

        flat_pos = skel_norm.reshape(skel_norm.shape[0], -1)
        key_joint_ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 19]
        key_pos_idx = []
        for j in key_joint_ids:
            key_pos_idx.extend([j*3, j*3+1, j*3+2])
        pos_feats = self._temporal_statistics(flat_pos[:, key_pos_idx])

        distances = self._compute_joint_distances(skel_norm)
        dist_feats = self._temporal_statistics(distances)

        angles = self._compute_joint_angles(skel_norm)
        angle_feats = self._temporal_statistics(angles)

        velocity = self._compute_velocity(skel_norm)
        vel_flat = velocity.reshape(velocity.shape[0], -1)
        vel_feats = self._temporal_statistics(vel_flat[:, key_pos_idx])

        accel = self._compute_acceleration(skel_norm)
        acc_flat = accel.reshape(accel.shape[0], -1)
        acc_feats = self._temporal_statistics(acc_flat[:, key_pos_idx])

        cov_feats = self._covariance_features(skel_norm)

        com = np.mean(skel_norm, axis=1)
        com_feats = self._temporal_statistics(com)

        if skel_norm.shape[0] > 1:
            total_disp = np.sum(np.linalg.norm(np.diff(skel_norm, axis=0), axis=2), axis=1)
        else:
            total_disp = np.array([0.0])
        motion_energy = np.array([
            np.mean(total_disp), np.std(total_disp), np.max(total_disp), np.sum(total_disp)
        ])

        all_feats = np.concatenate([
            pos_feats, dist_feats, angle_feats, vel_feats, acc_feats,
            cov_feats, com_feats, motion_energy,
        ])
        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

skel_extractor = SkeletonFeatureExtractor()
print("Skeleton feature extractor defined (identical to UTD-MHAD)")

Skeleton feature extractor defined (identical to UTD-MHAD)


## Depth Feature Extraction (identical to UTD-MHAD)

In [5]:
class DepthFeatureExtractor:
    """
    Extracts features from depth map sequences.
    IDENTICAL to UTD-MHAD pipeline.
    """

    def __init__(self, config):
        self.n_sample_frames = config.DEPTH_SAMPLE_FRAMES

    def _get_silhouette(self, depth_frame):
        valid = depth_frame[depth_frame > 0]
        if len(valid) == 0:
            return np.zeros_like(depth_frame, dtype=bool)
        threshold = np.percentile(valid, 50)
        return (depth_frame > 0) & (depth_frame < threshold)

    def _sobel_features(self, frame):
        sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
        sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)
        gx = convolve(frame.astype(np.float64), sobel_x)
        gy = convolve(frame.astype(np.float64), sobel_y)
        magnitude = np.sqrt(gx**2 + gy**2)
        direction = np.arctan2(gy, gx + 1e-12)
        features = [np.mean(magnitude), np.std(magnitude), np.max(magnitude),
                    np.sum(magnitude > np.mean(magnitude))]
        dir_hist, _ = np.histogram(direction[magnitude > np.mean(magnitude)],
                                    bins=8, range=(-np.pi, np.pi))
        if np.sum(dir_hist) > 0:
            dir_hist = dir_hist / (np.sum(dir_hist) + 1e-12)
        features.extend(dir_hist.tolist())
        mag_hist, _ = np.histogram(magnitude.ravel(), bins=8)
        if np.sum(mag_hist) > 0:
            mag_hist = mag_hist / (np.sum(mag_hist) + 1e-12)
        features.extend(mag_hist.tolist())
        return features

    def _shape_features(self, silhouette):
        features = []
        area = np.sum(silhouette)
        features.append(area)
        if area == 0:
            return features + [0] * 7
        rows = np.any(silhouette, axis=1)
        cols = np.any(silhouette, axis=0)
        if np.any(rows) and np.any(cols):
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            height = rmax - rmin + 1
            width = cmax - cmin + 1
            features.extend([height, width, height / (width + 1e-6), area / (height * width + 1e-6)])
        else:
            features.extend([0, 0, 0, 0])
        y_coords, x_coords = np.where(silhouette)
        if len(y_coords) > 0:
            features.extend([np.mean(y_coords) / silhouette.shape[0],
                           np.mean(x_coords) / silhouette.shape[1],
                           np.std(y_coords) / (silhouette.shape[0] + 1e-6)])
        else:
            features.extend([0, 0, 0])
        return features

    def _depth_distribution_features(self, depth_frame):
        valid = depth_frame[depth_frame > 0].ravel()
        if len(valid) == 0:
            return [0] * 8
        return [np.mean(valid), np.std(valid), np.median(valid), np.min(valid),
                np.max(valid), stats.skew(valid) if len(valid) > 2 else 0,
                stats.kurtosis(valid) if len(valid) > 3 else 0,
                np.percentile(valid, 75) - np.percentile(valid, 25)]

    def _extract_frame_features(self, frame):
        features = []
        scale = 4
        small = frame[::scale, ::scale]
        sil = self._get_silhouette(small)
        features.extend(self._shape_features(sil))
        features.extend(self._sobel_features(small))
        features.extend(self._depth_distribution_features(small))
        return features

    def extract(self, depth_data):
        if depth_data is None or depth_data.size == 0:
            return None
        depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ensure shape is (frames, height, width)
        if depth_data.ndim == 3:
            # MSR depth is already (frames, rows, cols) from our loader
            pass
        else:
            return None

        n_frames = depth_data.shape[0]
        if n_frames <= self.n_sample_frames:
            frame_indices = list(range(n_frames))
        else:
            frame_indices = np.linspace(0, n_frames - 1, self.n_sample_frames, dtype=int)

        frame_features = []
        for idx in frame_indices:
            ff = self._extract_frame_features(depth_data[idx])
            frame_features.append(ff)

        frame_features = np.array(frame_features)

        temporal_feats = []
        if frame_features.shape[0] > 1:
            diffs = np.diff(frame_features, axis=0)
            for col in range(diffs.shape[1]):
                temporal_feats.extend([np.mean(diffs[:, col]), np.std(diffs[:, col])])
        else:
            temporal_feats = [0.0] * (frame_features.shape[1] * 2)

        aggregated = []
        for col in range(frame_features.shape[1]):
            col_data = frame_features[:, col]
            aggregated.extend([np.mean(col_data), np.std(col_data),
                             np.min(col_data), np.max(col_data)])

        all_feats = np.concatenate([aggregated, temporal_feats])
        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

depth_extractor = DepthFeatureExtractor(config)
print("Depth feature extractor defined (identical to UTD-MHAD)")

Depth feature extractor defined (identical to UTD-MHAD)


## Feature Extraction Pipeline

In [6]:
def run_extraction():
    """Load data, extract features, save in UTD-MHAD compatible format."""

    print("=" * 70)
    print("MSR-ACTION3D FEATURE EXTRACTION PIPELINE")
    print("=" * 70)

    # Step 1: Load data
    print("\n--- Loading data ---")
    skel_data = loader.load_skeleton_data() if config.USE_SKELETON else []
    depth_data = loader.load_depth_data() if config.USE_DEPTH else []

    # Organize by sample key
    samples = {}
    for action, subject, episode, data in skel_data:
        key = f"a{action:02d}_s{subject:02d}_e{episode:02d}"
        if key not in samples:
            samples[key] = {'action': action, 'subject': subject, 'episode': episode}
        samples[key]['skeleton'] = data

    for action, subject, episode, data in depth_data:
        key = f"a{action:02d}_s{subject:02d}_e{episode:02d}"
        if key not in samples:
            samples[key] = {'action': action, 'subject': subject, 'episode': episode}
        samples[key]['depth'] = data

    print(f"Total unique samples: {len(samples)}")
    print(f"  With skeleton: {sum(1 for s in samples.values() if 'skeleton' in s)}")
    print(f"  With depth: {sum(1 for s in samples.values() if 'depth' in s)}")

    # Step 2: Extract features
    print("\n--- Extracting features ---")
    features_dict = {}
    total = len(samples)

    for i, (key, sample) in enumerate(sorted(samples.items())):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  Processing {i+1}/{total}...")

        feature_parts = []

        if config.USE_SKELETON and 'skeleton' in sample:
            skel_feats = skel_extractor.extract(sample['skeleton'])
            if skel_feats is not None:
                feature_parts.append(('skeleton', skel_feats))

        if config.USE_DEPTH and 'depth' in sample:
            depth_feats = depth_extractor.extract(sample['depth'])
            if depth_feats is not None:
                feature_parts.append(('depth', depth_feats))

        if feature_parts:
            features_dict[key] = {
                'label': sample['action'],
                'subject': sample['subject'],
                'modality_features': {name: f for name, f in feature_parts},
            }

    print(f"\nExtracted features for {len(features_dict)} / {total} samples")

    if features_dict:
        sample_entry = next(iter(features_dict.values()))
        for mod, feats in sample_entry['modality_features'].items():
            print(f"  {mod}: {len(feats)} features")

    # Step 3: Save in UTD-MHAD compatible format
    print("\n--- Saving features ---")
    out_dir = config.FEATURES_DIR
    os.makedirs(out_dir, exist_ok=True)

    keys = sorted(features_dict.keys())
    X_feat = []
    y_raw = []
    subjects_arr = []

    for k in keys:
        entry = features_dict[k]
        mod_feats = entry['modality_features']

        # Save in same dict format as UTD-MHAD (with empty arrays for missing modalities)
        sample_dict = {
            'skeleton_feat': mod_feats.get('skeleton', np.array([])),
            'sensor_feat': np.array([]),       # No inertial sensor in MSR-Action3D
            'depth_feat': mod_feats.get('depth', np.array([])),
            'video_feat': np.array([]),        # No RGB video
        }
        X_feat.append(sample_dict)
        y_raw.append(entry['label'])
        subjects_arr.append(entry['subject'])

    y_raw = np.array(y_raw)
    subjects_arr = np.array(subjects_arr)

    # Fit label encoder
    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    # Save
    joblib.dump(X_feat, os.path.join(out_dir, 'X_feat.pkl'))
    np.save(os.path.join(out_dir, 'y.npy'), y)
    np.save(os.path.join(out_dir, 'subjects.npy'), subjects_arr)
    joblib.dump(le, os.path.join(out_dir, 'label_encoder.pkl'))

    # Summary
    print(f"\nFeatures saved to '{out_dir}/':")
    print(f"  X_feat.pkl:        {len(X_feat)} samples")
    first = X_feat[0]
    for mod_key in ['skeleton_feat', 'sensor_feat', 'depth_feat', 'video_feat']:
        dim = first[mod_key].shape[0] if first[mod_key].size > 0 else 0
        print(f"    {mod_key}: {dim} features")
    total_dim = sum(first[m].shape[0] for m in first if first[m].size > 0)
    print(f"    TOTAL: {total_dim} features")
    print(f"  y.npy:             {y.shape} (encoded, {len(le.classes_)} classes)")
    print(f"  subjects.npy:      {subjects_arr.shape} "
          f"(subjects: {sorted(np.unique(subjects_arr).tolist())})")
    print(f"  label_encoder.pkl: classes = {le.classes_.tolist()}")

    # Print action name mapping
    print(f"\nAction label mapping:")
    for encoded_label, raw_label in enumerate(le.classes_):
        action_idx = int(raw_label)
        name = config.ACTION_NAMES[action_idx - 1] if action_idx <= len(config.ACTION_NAMES) else f"action_{action_idx}"
        print(f"  {encoded_label} -> raw {raw_label} -> {name}")

    return features_dict, X_feat, y, subjects_arr, le

print("Pipeline function defined")

Pipeline function defined


## Run Feature Extraction

In [7]:
features_dict, X_feat, y, subjects, le = run_extraction()

MSR-ACTION3D FEATURE EXTRACTION PIPELINE

--- Loading data ---


2026-03-20 13:10:24,476 - INFO - Loaded 567 skeleton sequences
2026-03-20 13:10:28,773 - INFO - Loaded 440 depth sequences


Total unique samples: 567
  With skeleton: 567
  With depth: 440

--- Extracting features ---
  Processing 1/567...
  Processing 50/567...
  Processing 100/567...
  Processing 150/567...
  Processing 200/567...
  Processing 250/567...
  Processing 300/567...
  Processing 350/567...
  Processing 400/567...
  Processing 450/567...
  Processing 500/567...
  Processing 550/567...

Extracted features for 567 / 567 samples
  skeleton: 1645 features
  depth: 216 features

--- Saving features ---

Features saved to 'features_new/':
  X_feat.pkl:        567 samples
    skeleton_feat: 1645 features
    sensor_feat: 0 features
    depth_feat: 216 features
    video_feat: 0 features
    TOTAL: 1861 features
  y.npy:             (567,) (encoded, 20 classes)
  subjects.npy:      (567,) (subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
  label_encoder.pkl: classes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Action label mapping:
  0 -> raw 1 -> high_arm_wave
  1 -> raw 2 -> h

## Verification

In [8]:
# Verify the saved features can be loaded back correctly
import joblib
import numpy as np
from pathlib import Path

FEATURE_DIR = Path(config.FEATURES_DIR)

X_feat_loaded = joblib.load(FEATURE_DIR / "X_feat.pkl")
y_loaded = np.load(FEATURE_DIR / "y.npy")
subjects_loaded = np.load(FEATURE_DIR / "subjects.npy")
le_loaded = joblib.load(FEATURE_DIR / "label_encoder.pkl")

print(f"Loaded {len(X_feat_loaded)} samples")
print(f"Number of classes: {len(np.unique(y_loaded))}")
print(f"Number of subjects: {len(np.unique(subjects_loaded))}")

first_sample = X_feat_loaded[0]
FEATURE_DIMS = {}
for key in ['depth_feat', 'sensor_feat', 'skeleton_feat', 'video_feat']:
    dim = first_sample[key].shape[0] if first_sample[key].size > 0 else 0
    FEATURE_DIMS[key.replace('_feat', '')] = dim
    print(f"  {key}: {dim}")
print(f"  TOTAL: {sum(FEATURE_DIMS.values())}")

print(f"\nClasses: {le_loaded.classes_.tolist()}")
print(f"Subjects: {sorted(np.unique(subjects_loaded).tolist())}")

# Samples per class
print(f"\nSamples per class:")
for cls_idx in range(len(le_loaded.classes_)):
    count = np.sum(y_loaded == cls_idx)
    raw_label = le_loaded.classes_[cls_idx]
    name = config.ACTION_NAMES[int(raw_label) - 1] if int(raw_label) <= len(config.ACTION_NAMES) else f"action_{raw_label}"
    print(f"  {cls_idx} ({name}): {count} samples")

print(f"\nSamples per subject:")
for subj in sorted(np.unique(subjects_loaded)):
    count = np.sum(subjects_loaded == subj)
    print(f"  Subject {subj}: {count} samples")

print("\nFeature extraction complete and verified!")

Loaded 567 samples
Number of classes: 20
Number of subjects: 10
  depth_feat: 216
  sensor_feat: 0
  skeleton_feat: 1645
  video_feat: 0
  TOTAL: 1861

Classes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Samples per class:
  0 (high_arm_wave): 27 samples
  1 (horizontal_arm_wave): 27 samples
  2 (hammer): 27 samples
  3 (hand_catch): 26 samples
  4 (forward_punch): 26 samples
  5 (high_throw): 26 samples
  6 (draw_x): 28 samples
  7 (draw_tick): 30 samples
  8 (draw_circle): 30 samples
  9 (hand_clap): 30 samples
  10 (two_hand_wave): 30 samples
  11 (side_boxing): 30 samples
  12 (bend): 30 samples
  13 (forward_kick): 30 samples
  14 (side_kick): 20 samples
  15 (jogging): 30 samples
  16 (tennis_swing): 30 samples
  17 (tennis_serve): 30 samples
  18 (golf_swing): 30 samples
  19 (pickup_throw): 30 samples

Samples per subject:
  Subject 1: 60 samples
  Subject 2: 59 samples
  Subject 3: 56 samples
  Subject 4: 